### Подготовка окружения в Google Colab

Я запускал этот проект в **Google Colab**, поэтому сначала подготовил окружение под PySpark:

- обновил пакеты в среде Colab;
- установил **JDK 17** (Spark работает поверх JVM);
- поставил **PySpark через pip** (`pyspark==3.5.1`). В этом варианте Spark подтягивается вместе с PySpark, поэтому мне **не пришлось** скачивать архив Spark вручную и подключать `findspark`.

Дальше я создаю `SparkSession` в режиме `local[*]` и проверяю версию Spark, чтобы убедиться, что всё поднялось корректно.


In [1]:
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==3.5.1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


Добавляем значения "JAVA_HOME" и "SPARK_HOME" в переменные среды с помощью метода "os.environ"   и указываем каталоги, где находятся Java и пакет Spark. Объявляем "Master" в нашем случае он локальный ("local[*]").


In [4]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("pet_project_create_mart")
         .getOrCreate())

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

# Проверка
spark.version


'3.5.1'

Устанавливаем и подключаем необходимые для выполнения задания библиотеки, описанные в README.md и подключаем Google Drive, чтобы сохранять сгенерированные файлы локально.

In [5]:
!pip install pdoc3
!pip install country_list
!pip install countryinfo

import random
import uuid
import string
import pyspark.sql.types as T
import pyspark.sql.functions as F
import countryinfo
import country_list
import hashlib
from  datetime import date
from  datetime import datetime
from  datetime import timedelta
import logging
from google.colab import drive

drive.mount("/content/gdrive")

logger = logging.getLogger()
logging.basicConfig(
      filename = "mylog.log",
      format = "%(asctime)s - %(levelname)s - %(funcName)s: %(lineno)d - %(message)s",
      datefmt='%H:%M:%S',
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 602.2/602.2 kB 31.3 MB/s eta 0:00:00
Mounted at /content/gdrive


In [6]:
countries = list(country_list.countries_for_language("en"))

Функция генерирующее число в строковом виде.

In [7]:
def generation_value(count: int = 10):
    """Генерация числа.

    Parameters:
        count: int
              Количество цифр в числе.

    Returns:
        string:
              Сгенерированное число.
    """

    return "".join(random.choices(string.digits, k=count))

In [8]:
print(generation_value.__doc__)

Генерация числа.

    Parameters:
        count: int
              Количество цифр в числе.

    Returns:
        string:
              Сгенерированное число.
    


In [9]:
pdoc generation_value

Создаём функцию "generate_rows_table", которая позволит сгенерировать список необходимых данных представленных для дальнейшего заполнения таблиц(витрин) из файла "Витрины.docx". Аргументами данной функции является число строк, которые мы хотим вставить в таблицу за раз и дата в поле "timestampcolumn".

# Новый раздел

In [14]:
def generate_rows_table(count: int = 1,
                    timestampcolumn: datetime = date.today()):
    """Генерация строк таблицы.

    Parameters:
        count: int
              Количество строк.
        timestampcolumn: datetime
              Дата в колонке 'timestampcolumn'.

    Returns:
        list_cookies: list
              Возвращает массив списков, представляющий собой строку таблицы.
    """
    list_cookies = list()
    for _ in range(count):
        inn = generation_value(12)

        _sa_cookie_a = {
            "key": "_sa_cookie_a",
            "value": f"SA1.{uuid.uuid4()}.{generation_value(10)}"
        }

        _fa_cookie_a = {
            "key": "_fa_cookie_a",
            "value": f"ads2.{generation_value(10)}.{generation_value(10)}"
        }

        _ym_cookie_c = {
            "key": "_ym_cookie_c",
            "value": {''.join(generation_value(20))}
        }

        _fbp = {
            "key": "_fbp",
            "value": f"fb.{random.choice(string.digits)}."
                         f"{generation_value(13)}."
                         f"{generation_value(9)}"
        }

        org_uid = {
            "key": "org_uid",
            "value": f"{generation_value(7)}"
        }

        user_uid = {
            "key": "user_uid",
            "value": f"{generation_value(7)}"
        }

        user_phone = {
            "key": "user_phone",
            "value": f"79{generation_value(2)}"
                     f"{generation_value(3)}"
                     f"{generation_value(2)}"
                     f"{generation_value(2)}"
        }

        user_mail = {
            "key": "user_mail",
            "value": f"""{"".join(random.choices(string.ascii_letters +
                             string.digits, k=10))}@user.io"""
        }

        event_type = random.choice(["SUBMIT", "REGISTER", "SUBMIT_MD5"])

        event_action = random.choice(["pageview", "event", "login-check-otp"])

        if event_type == "SUBMIT":
            data_value = hashlib.sha256(bytes(inn, encoding="utf-8")).hexdigest()
        elif event_type == "SUBMIT_MD5":
            data_value = hashlib.md5(bytes(inn, encoding="utf-8")).hexdigest()
        else:
            data_value = None

        geocountry = random.choice(countries)
        country_name = geocountry[1]

        try:
            country = countryinfo.CountryInfo(country_name)
            city = country.capital()
            geoaltitude = ",".join(map(str, country.latlng()))
        except KeyError:
            logger.warning(f"Unknown country in CountryInfo: {country_name}")
            city = None
            geoaltitude = None


        meta_platform = random.choice(["WEB", "MOBAIL"])

        if meta_platform == "WEB":
            user_os = random.choice(["Mac", "Windows", "Ubuntu"])
        else:
            user_os = random.choice(["IOS", "Android", "HarmonyOS", "BlackBerryOS"])
        systemlanguage = random.choice(["RU", "ENG"])

        screensize = "1920x1080"


        list_cookies.append([
            inn,
            [_sa_cookie_a, _fa_cookie_a, _ym_cookie_c, _fbp,
            org_uid, user_uid, user_phone, user_mail],
            event_type,
            event_action,
            data_value,
            geocountry[1],
            city,
            user_os,
            systemlanguage,
            geoaltitude,
            meta_platform,
            screensize,
            timestampcolumn
                             ])
    return list_cookies

Объявление схемы данных таблицы

In [15]:
schema = T.StructType([
    T.StructField("inn", T.StringType(), True),
    T.StructField("raw_cookie", T.ArrayType(T.MapType(T.StringType(),
                                                      T.StringType()))),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("event_action", T.StringType(), True),
    T.StructField("data_value", T.StringType(), True),
    T.StructField("geocountry", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("user_os", T.StringType(), True),
    T.StructField("systemlanguage", T.StringType(), True),
    T.StructField("geoaltitude", T.StringType(), True),
    T.StructField("meta_platform", T.StringType(), True),
    T.StructField("screensize", T.StringType(), True),
    T.StructField("timestampcolumn", T.DateType(), True)
                       ])

Цикл, генерируемый записи таблицы и сохраняемый их в файлы формата JSON, в
каталоги по дням недели. Все каталоги создаются автоматически. Данный цикл эмулирует постоянное добавление файлов. Необходимо запустить в фоновом режиме, пока идет работа с файлом "Create_data_marts.ipynb". Это требуется, чтобы смоделировать приближенную картину к реальности при работе с витриной "B" и витриной "D", так как они одинаковые, но количество строк в них может быть разным.

Чтобы не допустить переполнение диска, цикл отработает 10 раз после чего
выключится самостоятельно. По итогу будет сгеренировано тестовых данных за 10 дней.



In [17]:
import os

current_date = date.today()
folder = "/content/gdrive/MyDrive/data/json/"
if not os.path.isdir(folder):
    os.makedirs(folder)
i = 0
while i < 10:
    for _ in range(random.randrange(5, 10)):
        df = spark.createDataFrame(generate_rows_table(30, current_date),
                                   schema=schema)
        df.coalesce(1).write.mode("append") \
            .json(f"{folder}{current_date.strftime('%Y_%m_%d')}")

    current_date = current_date + timedelta(1)
    i = i + 1